# Calculating Kappa for capacitively coupled resonator to feedline
Using The SQUADDs paper https://arxiv.org/pdf/2312.13483 as source,

$C_r = \frac{\pi}{m \omega_r Z_0}$, where $m=2$ for $\lambda/2$ resonator and $m=4$ for $\lambda/4$ resonator.

 $Z_0=50 \Omega$ and $\kappa = \frac{\omega_r}{Q}$, where Q is the quality factor, and $Q = \frac{C_r}{\omega_r Z_0 C_c^2}$  

In [6]:
import numpy as np

In [7]:
freq_GHz = 12.5
m = 2
Z_0 = 50       # Ohms
Cc_fF = 13.4   # fF

In [8]:
def calc_cr(m, f_GHz, Z_0):
    omega_r = 2 * np.pi * f_GHz * 1e9   # rad/s
    return np.pi / (m * omega_r * Z_0)   # Farads

def calc_Q(f_GHz, m, Cc_fF, Z_0):
    omega_r = 2 * np.pi * f_GHz * 1e9   # rad/s
    Cc = Cc_fF * 1e-15                   # Farads
    Cr = calc_cr(m, f_GHz, Z_0)         # Farads
    return Cr / (omega_r * Z_0 * Cc**2) # dimensionless

def calc_kappa_kHz(f_GHz, m, Cc_fF, Z_0):
    omega_r = 2 * np.pi * f_GHz * 1e9   # rad/s
    Q = calc_Q(f_GHz, m, Cc_fF, Z_0)
    kappa_rad_s = omega_r / Q            # rad/s
    return kappa_rad_s / (2 * np.pi * 1e3)  # kHz


In [9]:
Cr   = calc_cr(m, freq_GHz, Z_0)
Q    = calc_Q(freq_GHz, m, Cc_fF, Z_0)
kHz  = calc_kappa_kHz(freq_GHz, m, Cc_fF, Z_0)

print(f"Cr    = {Cr*1e15:.4f} fF")
print(f"Q     = {Q:.2f}")
print(f"κ/2π  = {kHz:.4f} kHz")

Cr    = 400.0000 fF
Q     = 567.27
κ/2π  = 22035.3272 kHz


## Coupled model with C_rf and C_cg (loaded frequency and alternate kappa)

Model from the coupler/CPW-claw design notes:

$\omega_r = \sqrt{\frac{C_r}{C_r + C_{rf} + C_{cg}}}\,\omega'$

$\kappa = \frac{1}{2} Z_0 \omega_r^2 \frac{C_{rf}^2}{C_r + C_{rf} + C_{cg}}$

Here $\omega'$ is the uncoupled frequency (`freq_GHz`), $C_{rf}$ is resonator-to-feedline, and $C_{cg}$ is coupler-to-ground. `Crf_fF` and `Ccg_fF` are placeholders; set them to your design values.

In [10]:
# Inputs for the coupled model (from Q3D capacitance matrix, fcap12)
Crf_fF = 32.09076  # resonator-to-feedline capacitance, fF (cap_body_0 - cap_body_1)
Ccg_fF = 36.51643  # resonator-to-ground capacitance, fF (cap_body_0 - ground_main_plane)

omega_prime = 2 * np.pi * freq_GHz * 1e9          # uncoupled angular freq, rad/s
Cr_F  = calc_cr(m, freq_GHz, Z_0)                 # Farads
Crf_F = Crf_fF * 1e-15
Ccg_F = Ccg_fF * 1e-15
C_tot = Cr_F + Crf_F + Ccg_F

# Coupled (loaded) resonant frequency
omega_r_loaded = np.sqrt(Cr_F / C_tot) * omega_prime
f_r_loaded_GHz = omega_r_loaded / (2 * np.pi * 1e9)

# Linewidth from the image formula, converted to kappa/2pi in kHz
kappa_image_rad_s = 0.5 * Z_0 * omega_r_loaded**2 * Crf_F**2 / C_tot
kappa_image_kHz = kappa_image_rad_s / (2 * np.pi * 1e3)

# Notebook's Goppl-style kappa for comparison (uses C_c = Crf here)
kappa_notebook_kHz = calc_kappa_kHz(freq_GHz, m, Crf_fF, Z_0)

print(f"Cr                 = {Cr_F*1e15:.4f} fF")
print(f"f_r (uncoupled)    = {freq_GHz:.4f} GHz")
print(f"f_r (loaded)       = {f_r_loaded_GHz:.4f} GHz")
print(f"kappa/2pi (image)  = {kappa_image_kHz:.4f} kHz")
print(f"kappa/2pi (notebook, Goppl) = {kappa_notebook_kHz:.4f} kHz")
print(f"ratio notebook/image        = {kappa_notebook_kHz/kappa_image_kHz:.3f}")

Cr                 = 400.0000 fF
f_r (uncoupled)    = 12.5000 GHz
f_r (loaded)       = 11.5488 GHz
kappa/2pi (image)  = 46040.7091 kHz
kappa/2pi (notebook, Goppl) = 126377.5444 kHz
ratio notebook/image        = 2.745
